In [1]:
import time
import itertools
import pandas as pd
from entangled_engine.game_simulation import EntangledGameSimulation
from entangled_engine.greedy_agent import GreedyAgent
from entangled_engine.minimax_agent import MinimaxAgent

def run_single_game(target_score, minimax_params, p1_agent_type='minimax', p2_agent_type='greedy'):
    """Simulates a single match and tracks total time spent on Minimax decisions."""
    sim = EntangledGameSimulation(target_score=target_score)
    total_minimax_time = 0
    minimax_move_count = 0

    while True:
        game_over, winner, _ = sim.check_game_over(check_stuck=True)
        if game_over:
            return winner, total_minimax_time, minimax_move_count

        current_p = sim.current_player

        if current_p == 'p1' and p1_agent_type == 'minimax':
            t0 = time.time()
            chosen_pair, m1, m2 = MinimaxAgent.select_best_move(
                sim,
                depth=minimax_params['depth'],
                top_k=minimax_params['top_k'],
                score_weight=minimax_params['score_weight'],
                mobility_weight=minimax_params['mobility_weight']
            )
            t1 = time.time()
            total_minimax_time += (t1 - t0)
            minimax_move_count += 1

        elif current_p == 'p2' and p2_agent_type == 'greedy':
            chosen_pair, m1, m2 = GreedyAgent.select_best_move(sim)
        else:
            valid_pairs = sim.board.activePairs(current_p)
            chosen_pair = valid_pairs[0] if valid_pairs else None
            m1, m2 = None, None

        if not chosen_pair or not m1 or not m2:
            # Player is stuck
            sim.switch_player()
            continue

        sim.execute_turn(chosen_pair, m1, m2)
        sim.switch_player()

def optimize_parameters(games_per_config=50, target_score=3):
    """Performs Grid Search over Minimax hyperparameters to find the highest win rate

    and lowest decision latency configuration.
    """
    # Define Parameter Grid Search Space
    param_grid = {
        'depth': [2, 3],
        'top_k': [4, 6, 8],
        'score_weight': [100, 1000],
        'mobility_weight': [1, 5, 10]
    }

    keys, values = zip(*param_grid.items())
    all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    print(f"🚀 Starting Minimax Hyperparameter Optimization Tournament...")
    print(f"📊 Total Configurations to Test: {len(all_combinations)}")
    print(f"🎮 Games per Configuration: {games_per_config} | Target Score: {target_score}\n" + "-"*75)

    results = []

    for idx, params in enumerate(all_combinations, 1):
        p1_wins = 0
        p2_wins = 0
        draws = 0
        config_time = 0
        config_moves = 0

        for game_i in range(games_per_config):
            winner, g_time, g_moves = run_single_game(target_score, params)
            config_time += g_time
            config_moves += g_moves

            if winner == 'p1':
                p1_wins += 1
            elif winner == 'p2':
                p2_wins += 1
            else:
                draws += 1

        win_rate = (p1_wins / games_per_config) * 100
        avg_move_time_ms = (config_time / max(1, config_moves)) * 1000

        res_entry = {
            'Config_ID': idx,
            'Depth': params['depth'],
            'Top_K': params['top_k'],
            'Score_Weight': params['score_weight'],
            'Mobility_Weight': params['mobility_weight'],
            'P1_Win_%': round(win_rate, 2),
            'P2_Win_%': round((p2_wins / games_per_config) * 100, 2),
            'Draw_%': round((draws / games_per_config) * 100, 2),
            'Avg_Turn_Time_ms': round(avg_move_time_ms, 2)
        }
        results.append(res_entry)

        print(f"Config {idx:02d}/{len(all_combinations)} | Depth: {params['depth']} | Top_K: {params['top_k']} | "
              f"Score_W: {params['score_weight']:4d} | Mob_W: {params['mobility_weight']:2d} => "
              f"Win Rate: {win_rate:5.1f}% | Avg Move Time: {avg_move_time_ms:6.2f} ms")

    # Create Summary DataFrame
    df_results = pd.DataFrame(results)
    df_sorted = df_results.sort_values(by=['P1_Win_%', 'Avg_Turn_Time_ms'], ascending=[False, True])

    print("\n" + "="*75)
    print("🏆 TOP 5 OPTIMAL MINIMAX CONFIGURATIONS 🏆")
    print("="*75)
    print(df_sorted.head(5).to_string(index=False))

    optimal_config = df_sorted.iloc[0]
    print("\n🎯 ABSOLUTE BEST PARAMETER SET:")
    print(f" - Optimal Depth:           {int(optimal_config['Depth'])}")
    print(f" - Optimal Top_K:           {int(optimal_config['Top_K'])}")
    print(f" - Optimal Score Weight:    {int(optimal_config['Score_Weight'])}")
    print(f" - Optimal Mobility Weight: {int(optimal_config['Mobility_Weight'])}")
    print(f" - Peak Win Rate vs Greedy: {optimal_config['P1_Win_%']}%")
    print(f" - Avg Latency per Move:    {optimal_config['Avg_Turn_Time_ms']} ms")

    return df_sorted

# Run Optimizer
if __name__ == "__main__":
    df_summary = optimize_parameters(games_per_config=50, target_score=3)

🚀 Starting Minimax Hyperparameter Optimization Tournament...
📊 Total Configurations to Test: 36
🎮 Games per Configuration: 50 | Target Score: 3
---------------------------------------------------------------------------
Config 01/36 | Depth: 2 | Top_K: 4 | Score_W:  100 | Mob_W:  1 => Win Rate: 100.0% | Avg Move Time: 123.55 ms
Config 02/36 | Depth: 2 | Top_K: 4 | Score_W:  100 | Mob_W:  5 => Win Rate: 100.0% | Avg Move Time: 118.15 ms
Config 03/36 | Depth: 2 | Top_K: 4 | Score_W:  100 | Mob_W: 10 => Win Rate: 100.0% | Avg Move Time: 111.31 ms
Config 04/36 | Depth: 2 | Top_K: 4 | Score_W: 1000 | Mob_W:  1 => Win Rate: 100.0% | Avg Move Time: 124.33 ms
Config 05/36 | Depth: 2 | Top_K: 4 | Score_W: 1000 | Mob_W:  5 => Win Rate: 100.0% | Avg Move Time: 113.41 ms
Config 06/36 | Depth: 2 | Top_K: 4 | Score_W: 1000 | Mob_W: 10 => Win Rate: 100.0% | Avg Move Time: 115.04 ms
Config 07/36 | Depth: 2 | Top_K: 6 | Score_W:  100 | Mob_W:  1 => Win Rate: 100.0% | Avg Move Time: 151.39 ms
Config 08/

KeyboardInterrupt: 

In [ ]:
import time
import itertools
import random

import pandas as pd

from entangled_engine.game_simulation import EntangledGameSimulation
from entangled_engine.greedy_agent import GreedyAgent
from entangled_engine.minimax_agent import MinimaxAgent


def run_single_game(target_score, minimax_params, seed=None,
                     p1_agent_type="minimax", p2_agent_type="greedy"):
  """Simulates one game and returns (winner, reason, total_minimax_decision_time, minimax_move_count).

  execute_turn() already switches the current player internally at the end
  of every turn -- do NOT call sim.switch_player() again here, that was
  the bug that made p2 never get a turn in the earlier version of this
  script (and is why every earlier config showed exactly 100% win rate).
  """
  if seed is not None:
    random.seed(seed)

  sim = EntangledGameSimulation(
      p1_type=p1_agent_type, p2_type=p2_agent_type, target_score=target_score
  )
  total_minimax_time = 0.0
  minimax_move_count = 0

  while True:
    game_over, winner, reason = sim.check_game_over(check_stuck=True)
    if game_over:
      return winner, reason, total_minimax_time, minimax_move_count

    current_p = sim.current_player
    agent_type = sim.p1_type if current_p == "p1" else sim.p2_type

    if agent_type == "minimax":
      t0 = time.time()
      chosen_pair, m1, m2 = MinimaxAgent.select_best_move(
          sim,
          depth=minimax_params["depth"],
          top_k=minimax_params["top_k"],
          score_weight=minimax_params["score_weight"],
          mobility_weight=minimax_params["mobility_weight"],
      )
      total_minimax_time += time.time() - t0
      minimax_move_count += 1
    elif agent_type == "greedy":
      chosen_pair, m1, m2 = GreedyAgent.select_best_move(sim)
    else:  # random fallback, e.g. for cross-validation matches
      valid_pairs = sim.board.activePairs(current_p)
      chosen_pair = random.choice(valid_pairs)
      m1, m2 = None, None

    sim.execute_turn(chosen_pair, m1, m2)


def evaluate_config(params, games_per_config, target_score, base_seed,
                     p1_agent_type="minimax", p2_agent_type="greedy"):
  p1_wins = p2_wins = draws = 0
  total_time = 0.0
  total_moves = 0

  for i in range(games_per_config):
    winner, _reason, g_time, g_moves = run_single_game(
        target_score, params, seed=base_seed + i,
        p1_agent_type=p1_agent_type, p2_agent_type=p2_agent_type,
    )
    total_time += g_time
    total_moves += g_moves
    if winner == "p1":
      p1_wins += 1
    elif winner == "p2":
      p2_wins += 1
    else:
      draws += 1

  win_rate = (p1_wins / games_per_config) * 100
  avg_move_time_ms = (total_time / max(1, total_moves)) * 1000

  return {
      "Depth": params["depth"],
      "Top_K": params["top_k"],
      "Score_Weight": params["score_weight"],
      "Mobility_Weight": params["mobility_weight"],
      "P1_Win_%": round(win_rate, 2),
      "P2_Win_%": round((p2_wins / games_per_config) * 100, 2),
      "Draw_%": round((draws / games_per_config) * 100, 2),
      "Avg_Turn_Time_ms": round(avg_move_time_ms, 2),
  }


def optimize_parameters(games_per_config=50, target_score=3, param_grid=None,
                         base_seed=1, opponent="greedy"):
  """Grid search over minimax hyperparameters vs. a fixed opponent type.

  depth=3 is left out of the default grid: it costs roughly 5-6x more per
  move than depth=2 for this game (measured separately), and against
  GreedyAgent, depth=2 already saturates near the win-rate ceiling -- so a
  broad sweep at depth=3 mostly burns time without changing the answer.
  Test depth=3 in a focused follow-up run (see run_depth3_check below)
  once you know which top_k/weights already win at depth=2.
  """
  if param_grid is None:
    param_grid = {
        "depth": [2],
        "top_k": [4, 6, 8],
        "score_weight": [100, 1000],
        "mobility_weight": [1, 5, 10],
    }

  keys, values = zip(*param_grid.items())
  all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

  print("🚀 Starting Minimax Hyperparameter Optimization Tournament...")
  print(f"📊 Total Configurations to Test: {len(all_combinations)}")
  print(f"🎮 Games per Configuration: {games_per_config} | Target Score: {target_score} "
        f"| Opponent: {opponent}\n" + "-" * 75)

  results = []
  for idx, params in enumerate(all_combinations, 1):
    t0 = time.time()
    res = evaluate_config(
        params, games_per_config, target_score, base_seed,
        p1_agent_type="minimax", p2_agent_type=opponent,
    )
    res["Config_ID"] = idx
    results.append(res)
    elapsed = time.time() - t0
    print(f"Config {idx:02d}/{len(all_combinations)} | Depth: {params['depth']} | "
          f"Top_K: {params['top_k']} | Score_W: {params['score_weight']:4d} | "
          f"Mob_W: {params['mobility_weight']:2d} => "
          f"Win Rate: {res['P1_Win_%']:5.1f}% | Avg Move Time: {res['Avg_Turn_Time_ms']:6.2f} ms "
          f"[{elapsed:.1f}s / {games_per_config} games]")

  df_results = pd.DataFrame(results)
  df_sorted = df_results.sort_values(
      by=["P1_Win_%", "Avg_Turn_Time_ms"], ascending=[False, True]
  )

  print("\n" + "=" * 75)
  print("🏆 TOP 5 CONFIGURATIONS 🏆")
  print("=" * 75)
  print(df_sorted.head(5).to_string(index=False))

  return df_sorted


def cross_validate_top_config(df_sorted, games_per_config=100, target_score=3,
                               base_seed=90000):
  """Re-tests the #1 config from the grid search against a DIFFERENT
  opponent (random) than it was tuned on (greedy). A config that only
  looks good against greedy's specific one-ply heuristic, rather than
  good in general, is a real risk with this kind of tuning -- this check
  is what would catch that."""
  top = df_sorted.iloc[0]
  params = {
      "depth": int(top["Depth"]),
      "top_k": int(top["Top_K"]),
      "score_weight": int(top["Score_Weight"]),
      "mobility_weight": int(top["Mobility_Weight"]),
  }

  print("\n" + "=" * 75)
  print("🔬 CROSS-VALIDATION: top config vs. an opponent it was NOT tuned on (random)")
  print("=" * 75)
  res = evaluate_config(
      params, games_per_config, target_score, base_seed,
      p1_agent_type="minimax", p2_agent_type="random",
  )
  print(f"vs random -> Win Rate: {res['P1_Win_%']}% | Draw: {res['Draw_%']}% | "
        f"Avg Move Time: {res['Avg_Turn_Time_ms']} ms")
  return res


def run_depth3_check(best_config, games_per_config=30, target_score=3,
                      base_seed=70000, opponent="greedy"):
  """Focused follow-up: is depth=3 worth its cost for the ALREADY-best
  depth=2 weights, rather than re-running the full grid at depth=3."""
  params = dict(best_config)
  params["depth"] = 3
  print("\n" + "=" * 75)
  print(f"🔎 Depth=3 check (top_k={params['top_k']}, score_w={params['score_weight']}, "
        f"mob_w={params['mobility_weight']}) vs {opponent}")
  print("=" * 75)
  res = evaluate_config(
      params, games_per_config, target_score, base_seed,
      p1_agent_type="minimax", p2_agent_type=opponent,
  )
  print(f"depth=3 -> Win Rate: {res['P1_Win_%']}% | Avg Move Time: {res['Avg_Turn_Time_ms']} ms")
  return res


if __name__ == "__main__":
  df_summary = optimize_parameters(games_per_config=50, target_score=3)

  top = df_summary.iloc[0]
  print("\n🎯 BEST PARAMETER SET (vs greedy):")
  print(f" - Depth:           {int(top['Depth'])}")
  print(f" - Top_K:           {int(top['Top_K'])}")
  print(f" - Score Weight:    {int(top['Score_Weight'])}")
  print(f" - Mobility Weight: {int(top['Mobility_Weight'])}")
  print(f" - Win Rate vs Greedy: {top['P1_Win_%']}%")
  print(f" - Avg Latency/Move:   {top['Avg_Turn_Time_ms']} ms")

  cross_validate_top_config(df_summary, games_per_config=100, target_score=3)

  run_depth3_check(
      {"top_k": int(top["Top_K"]), "score_weight": int(top["Score_Weight"]),
       "mobility_weight": int(top["Mobility_Weight"])},
      games_per_config=30, target_score=3,
  )